# 🧪 AI-Scientist-v2 on Kaggle

**Fully Automated Scientific Discovery via Agentic Tree Search**  
*Powered by DeepSeek V4 Flash 🚀*

This notebook runs [AI-Scientist-v2](https://github.com/SakanaAI/AI-Scientist_v2) by SakanaAI on Kaggle's free GPU (NVIDIA T4).

### Pipeline Overview:
1. **Setup** — Clone repo, install dependencies
2. **Ideation** — Generate research ideas via LLM
3. **Experimentation** — Agentic tree search runs experiments
4. **Write-up** — Generate a scientific paper with LaTeX
5. **Review** — LLM self-review of the paper

> ⚠️ **Warning:** This code executes LLM-written code. Run in a sandboxed environment.
> 
> 💰 **Cost:** ~$1-3 per run with **DeepSeek V4 Flash** (very affordable!). Kaggle GPU is **free**.

---
## 1. ⚙️ Setup Environment

Run the cell below to:
- Clone the AI-Scientist-v2 repository
- Install system packages (LaTeX, poppler, chktex)
- Install Python dependencies
- Verify GPU availability

In [ ]:
import os, sys, subprocess, json, shutil, time
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
PROJECT_DIR = WORKING_DIR / "AI-Scientist-v2"
os.chdir(WORKING_DIR)

print("=" * 60)
print("🚀 Starting AI-Scientist-v2 Setup on Kaggle")
print("=" * 60)

# ── Clone repo (your fork with DeepSeek V4 Flash support) ──
REPO_URL = "https://github.com/brahimje/AI-Scientist-v2"

if not PROJECT_DIR.exists():
    print(f"\n[1/6] Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print("\n[1/6] Repository already exists, pulling latest...")
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}

# ── Install system packages (minimal LaTeX for speed) ──
print("\n[2/6] Installing system packages (minimal LaTeX, poppler)...")
import subprocess as sp
import os
env = os.environ.copy()
env["DEBIAN_FRONTEND"] = "noninteractive"

cmds = [
    "apt-get update -qq",
    "apt-get install -y -qq --no-install-recommends texlive-latex-base texlive-latex-extra texlive-fonts-recommended texlive-bibtex-extra poppler-utils cm-super 2>&1",
]
for cmd in cmds:
    result = sp.run(cmd, shell=True, env=env, capture_output=True, text=True, timeout=600)
    if result.returncode != 0:
        print(f"   ⚠️ Warning (non-fatal): {result.stderr[-200:]}")
    else:
        print(f"   ✅ Package installed")

print("   ✅ System packages ready")

# ── Install Python dependencies ──
print("\n[3/6] Installing Python dependencies...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install -q -r requirements.txt
print("    ✅ Python packages installed")

# ── Verify GPU ──
print("\n[4/6] Verifying GPU...")
import torch
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    for i in range(gpu_count):
        print(f"    ✅ GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"       VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
else:
    print("    ⚠️ No GPU detected. CPU-only mode (slow for experiments).")

# ── Verify Python ──
print(f"\n[5/6] Python version: {sys.version}")

# ── Create directories ──
print("\n[6/6] Creating project directories...")
os.makedirs(PROJECT_DIR / "ai_scientist" / "ideas", exist_ok=True)
os.makedirs(PROJECT_DIR / "experiments", exist_ok=True)
os.makedirs(PROJECT_DIR / "logs", exist_ok=True)
print("    ✅ Directories created")

print("\n" + "=" * 60)
print("✅ Setup complete!")
print("=" * 60)

---
## 2. 🔑 Configure API Keys

Set your API keys as **Kaggle Secrets** (`Add-ons > Secrets` in the sidebar):

| Secret Name | Description | Required? |
|---|---|---|
| `DEEPSEEK_API_KEY` | DeepSeek API key (for `deepseek-v4-flash`) | ✅ **Yes** |
| `S2_API_KEY` | Semantic Scholar API key (for literature search) | ⚠️ Optional but **recommended** |
| `OPENAI_API_KEY` | OpenAI API key (GPT-4o, o1, o3) | ⚠️ Optional (fallback) |
| `ANTHROPIC_API_KEY` | Anthropic API key (Claude 3.5 Sonnet) | ⚠️ Optional |

> 💡 **DeepSeek V4 Flash** is ~10x cheaper than GPT-4o!  
> Get your DeepSeek key: [https://platform.deepseek.com/api_keys](https://platform.deepseek.com/api_keys)  
> Get a **free** Semantic Scholar key: [https://www.semanticscholar.org/product/api](https://www.semanticscholar.org/product/api)  
> _(Without S2_API_KEY, literature searches will be rate-limited but won't block the pipeline)_

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# ── Load API Keys from Kaggle Secrets ──
api_keys = {
    "DEEPSEEK_API_KEY": None,
    "OPENAI_API_KEY": None,
    "ANTHROPIC_API_KEY": None,
    "S2_API_KEY": None,
}

for key in api_keys:
    try:
        val = secrets.get_secret(key)
        if val:
            os.environ[key] = val
            api_keys[key] = "✅ Set (hidden)"
            print(f"    ✅ {key} loaded from Kaggle Secrets")
        else:
            api_keys[key] = "❌ Empty"
            print(f"    ⚠️ {key} is empty")
    except:
        api_keys[key] = "❌ Not set"
        print(f"    ⚠️ {key} not found in Kaggle Secrets")

print("\n" + "─" * 40)
if api_keys["DEEPSEEK_API_KEY"]:
    print("✅ DeepSeek API key is set! Ready to use deepseek-v4-flash.")
else:
    print("⚠️ Please add DEEPSEEK_API_KEY as a Kaggle Secret, then re-run this cell.")
    print("   Get one at: https://platform.deepseek.com/api_keys")

---
## 3. 💡 Generate Research Ideas (Ideation)

Create a topic description file and run the ideation script to generate research ideas.

**Edit the topic below** to match your research area of interest!

In [ ]:
%cd {PROJECT_DIR}

# ── Create a custom research topic description ──
# ✏️ EDIT THIS to describe YOUR research area
TOPIC_TITLE = "AI-Driven Cybersecurity: Threat Detection and Response"
TOPIC_KEYWORDS = "cybersecurity, intrusion detection, anomaly detection, AI security, threat intelligence"
TOPIC_TLDR = "Leveraging machine learning and deep learning to detect, classify, and respond to cybersecurity threats in real-time network traffic and system logs."
TOPIC_ABSTRACT = (
    "The increasing sophistication and frequency of cyberattacks pose significant "
    "challenges to traditional security mechanisms. Machine learning (ML) and deep "
    "learning (DL) have emerged as promising tools for enhancing cybersecurity through "
    "automated threat detection, anomaly identification, and real-time response. "
    "This work investigates novel approaches to apply AI techniques across multiple "
    "cybersecurity domains, including network intrusion detection, malware classification, "
    "phishing detection, and user behavior analytics. Key challenges such as imbalanced "
    "datasets, adversarial robustness, concept drift, and model interpretability are "
    "addressed. We explore hybrid models combining supervised and unsupervised learning, "
    "transformer-based architectures for log analysis, and reinforcement learning for "
    "adaptive defense mechanisms. The goal is to develop practical, deployable AI systems "
    "that improve security posture while maintaining low false-positive rates and "
    "computational efficiency suitable for real-world environments."
)

topic_md = f"""# Title: {TOPIC_TITLE}

## Keywords
{TOPIC_KEYWORDS}

## TL;DR
{TOPIC_TLDR}

## Abstract
{TOPIC_ABSTRACT}
"""

topic_file = PROJECT_DIR / "ai_scientist" / "ideas" / "my_research_topic.md"
topic_file.write_text(topic_md)

print(f"✅ Created topic file: {topic_file}")
print(f"\n📌 Topic: {TOPIC_TITLE}")
print(f"📌 Keywords: {TOPIC_KEYWORDS}")

In [ ]:
%cd {PROJECT_DIR}

# ── Run Ideation ──
# This generates research ideas using an LLM
# ⏱️ Expected time: ~10-30 minutes depending on model

IDEATION_MODEL = "deepseek-v4-flash"  # 🚀 Fast & cheap! Use "gpt-4o-mini" as fallback.
NUM_IDEAS = 3                         # Number of ideas to generate (start small for testing)
NUM_REFLECTIONS = 3                   # Refinement steps per idea

print(f"🚀 Running ideation with {IDEATION_MODEL}...")
print(f"   Generating {NUM_IDEAS} ideas with {NUM_REFLECTIONS} reflections each")
print()

!python ai_scientist/perform_ideation_temp_free.py \
    --workshop-file "ai_scientist/ideas/my_research_topic.md" \
    --model {IDEATION_MODEL} \
    --max-num-generations {NUM_IDEAS} \
    --num-reflections {NUM_REFLECTIONS} \
    2>&1

print("\n" + "─" * 40)
print("✅ Ideation complete!")

# Check the generated ideas file
ideas_file = PROJECT_DIR / "ai_scientist" / "ideas" / "my_research_topic.json"
if ideas_file.exists():
    with open(ideas_file) as f:
        ideas = json.load(f)
    if isinstance(ideas, list):
        print(f"\n📋 Generated {len(ideas)} ideas:")
        for i, idea in enumerate(ideas):
            print(f"   {i+1}. {idea.get('Title', idea.get('title', 'Untitled'))}")
    else:
        print(f"   ✅ Ideas saved to {ideas_file}")
else:
    print(f"   ⚠️ Ideas file not found at {ideas_file}")
    print("   Check the output above for errors.")

---
## 4. 🧪 Run Main Experiment Pipeline

This launches the AI-Scientist-v2 agentic tree search:
- **Stage 1:** Initial code generation & experiments
- **Stage 2-4:** Iterative improvement & debugging
- **Write-up:** Generates a LaTeX paper with plots
- **Review:** LLM self-review of the paper

> ⏱️ **Expected time:** 2-10+ hours depending on complexity and `steps` config.
> 
> 💾 **Checkpoints:** Results are saved automatically to `experiments/` folder.

In [ ]:
%cd {PROJECT_DIR}

# ── Verify ideas file exists ──
ideas_file = PROJECT_DIR / "ai_scientist" / "ideas" / "my_research_topic.json"
if not ideas_file.exists():
    # Try the default example idea
    ideas_file = PROJECT_DIR / "ai_scientist" / "ideas" / "i_cant_believe_its_not_better.json"
    
if not ideas_file.exists():
    print("❌ No ideas file found! Run the ideation step first.")
else:
    print(f"✅ Using ideas file: {ideas_file}")
    
    # Quick preview
    with open(ideas_file) as f:
        ideas_data = json.load(f)
    if isinstance(ideas_data, list):
        print(f"   Found {len(ideas_data)} idea(s)")
        for i, idea in enumerate(ideas_data):
            print(f"   Idea {i}: {idea.get('Title', idea.get('title', 'N/A'))}")
    else:
        print(f"   Loaded idea data")

In [ ]:
%cd {PROJECT_DIR}

# ── Configure Models ──
# 🚀 DeepSeek V4 Flash is the default - very affordable!
#    (~$0.15/M tokens input, $0.30/M tokens output - 10x cheaper than GPT-4o)

EXPERIMENT_MODEL = "deepseek-v4-flash"   # Model for experiments (cheap & capable)
WRITEUP_MODEL = "deepseek-v4-flash"      # Model for paper write-up
REVIEW_MODEL = "deepseek-v4-flash"       # Model for paper review
CITATION_MODEL = "deepseek-v4-flash"     # Model for citation gathering
PLOT_MODEL = "deepseek-v4-flash"         # Model for plot aggregation

# 💡 Fallback options (if you have other API keys):
# EXPERIMENT_MODEL = "claude-3-5-sonnet-20241022"  # Best quality (needs ANTHROPIC_API_KEY)
# WRITEUP_MODEL = "gpt-4o-2024-11-20"              # Alternative (needs OPENAI_API_KEY)

IDEA_IDX = 0                       # Which idea index to run (0 = first idea)
NUM_CITE_ROUNDS = 5                # Citation rounds (reduce for faster runs)

print("=" * 60)
print("🚀 Launching AI-Scientist-v2 Pipeline with DeepSeek V4 Flash")
print("=" * 60)
print(f"📌 Experiment model: {EXPERIMENT_MODEL}")
print(f"📌 Write-up model:   {WRITEUP_MODEL}")
print(f"📌 Review model:     {REVIEW_MODEL}")
print(f"📌 Idea index:       {IDEA_IDX}")
print(f"📌 Citation rounds:  {NUM_CITE_ROUNDS}")
print("=" * 60)

# ── Launch the pipeline ──
!python launch_scientist_bfts.py \
    --load_ideas "ai_scientist/ideas/my_research_topic.json" \
    --idea_idx {IDEA_IDX} \
    --model_writeup {WRITEUP_MODEL} \
    --model_citation {CITATION_MODEL} \
    --model_review {REVIEW_MODEL} \
    --model_agg_plots {PLOT_MODEL} \
    --num_cite_rounds {NUM_CITE_ROUNDS} \
    2>&1

print("\n" + "=" * 60)
print("✅ Pipeline finished!")
print("=" * 60)

---
## 5. 📊 Results & Export

After the pipeline completes, find your results in the `experiments/` folder.

### Expected output structure:
```
experiments/
  └── TIMESTAMP_IDEANAME/
      ├── TIMESTAMP_IDEANAME.pdf       ← 📄 Final paper!
      ├── logs/
      │   └── 0-run/
      │       └── unified_tree_viz.html ← 🌳 Tree visualization
      ├── workspaces/                    ← Experiment code & data
      └── token_tracker.json             ← 💰 Token usage
```

In [ ]:
%cd {PROJECT_DIR}

import glob

print("=" * 60)
print("📂 Results Summary")
print("=" * 60)

# ── Find generated PDFs ──
pdfs = list(PROJECT_DIR.glob("experiments/**/*.pdf"))
if pdfs:
    print(f"\n📄 Generated PDFs ({len(pdfs)} found):")
    for pdf in sorted(pdfs):
        size_mb = pdf.stat().st_size / 1e6
        print(f"   ✅ {pdf.relative_to(PROJECT_DIR)} ({size_mb:.1f} MB)")
else:
    print("\n   No PDFs found yet.")

# ── Find tree visualizations ──
viz_files = list(PROJECT_DIR.glob("experiments/**/unified_tree_viz.html"))
if viz_files:
    print(f"\n🌳 Tree visualizations ({len(viz_files)} found):")
    for viz in sorted(viz_files):
        print(f"   ✅ {viz.relative_to(PROJECT_DIR)}")

# ── Find experiments folder ──
exp_dirs = [d for d in (PROJECT_DIR / "experiments").iterdir() if d.is_dir()] if (PROJECT_DIR / "experiments").exists() else []
if exp_dirs:
    print(f"\n📁 Experiment directories ({len(exp_dirs)} found):")
    for d in sorted(exp_dirs):
        print(f"   📁 {d.name}")
        # Check for sub-items
        items = list(d.iterdir())
        for item in items[:10]:
            tag = "📁" if item.is_dir() else "📄"
            print(f"      {tag} {item.name}")
        if len(items) > 10:
            print(f"      ... and {len(items)-10} more items")

# ── Show token usage ──
token_files = list(PROJECT_DIR.glob("experiments/**/token_tracker.json"))
if token_files:
    print(f"\n💰 Token usage:")
    for tf in sorted(token_files):
        try:
            with open(tf) as f:
                tokens = json.load(f)
            print(f"   {tf.parent.name}: {json.dumps(tokens, indent=2)}")
        except:
            pass

print("\n" + "─" * 40)
print("💡 Tip: Download results from Kaggle")
print("   Go to the Data tab on the right → 'Output' → Download all")

---
## 6. 🚀 Performance Tips

### With DeepSeek V4 Flash (💰 ~$1-3 per run):
| Setting | Recommended | Impact |
|---------|-------------|--------|
| Experiment model | `deepseek-v4-flash` | Cheap & capable ✅ |
| Write-up model | `deepseek-v4-flash` | ~$0.30/M tokens output |
| `bfts_config.yaml > agent > num_workers` | `2` | Less parallelism, less memory |
| `bfts_config.yaml > agent > steps` | `3` | Fewer iterations, faster |
| `bfts_config.yaml > search > num_drafts` | `1` | Single tree, less exploration |
| `--num_cite_rounds` | `5-10` | Fewer citations, faster |

### For best quality (💰 $15-25 per run):
| Setting | Recommended |
|---------|-------------|
| Experiment model | `claude-3-5-sonnet-20241022` (best) or `gpt-4o` |
| Write-up model | `deepseek-v4-flash` (budget) or `gpt-4o` (quality) |
| `bfts_config.yaml > agent > num_workers` | `4` |
| `bfts_config.yaml > agent > steps` | `5` |

### Kaggle GPU limits:
- **30 hours/week** of GPU usage (free tier)
- Session disconnects after **~9 hours** of inactivity
- **Save checkpoints** to `/kaggle/working/` (persistent)

---
## 7. 🔄 Resume from Checkpoint

If your session disconnects, run this cell to restore your progress:

In [ ]:
import os, sys
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
PROJECT_DIR = WORKING_DIR / "AI-Scientist-v2"

if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)
    print(f"✅ Working directory: {PROJECT_DIR}")
    
    # Check for existing experiments
    exp_dir = PROJECT_DIR / "experiments"
    if exp_dir.exists():
        dirs = [d for d in exp_dir.iterdir() if d.is_dir()]
        print(f"📁 Found {len(dirs)} existing experiment(s):")
        for d in sorted(dirs):
            print(f"   📁 {d.name}")
    else:
        print("📁 No experiments found yet.")
else:
    print("❌ Project directory not found. Please re-run the setup cell.")

# Set up API keys again if needed
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ["DEEPSEEK_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "S2_API_KEY"]:
        try:
            val = secrets.get_secret(key)
            if val:
                os.environ[key] = val
        except:
            pass
    print("🔑 API keys restored from Kaggle Secrets")
except:
    print("⚠️ Could not load Kaggle Secrets (expected on first run)")

---
## 📖 Additional Resources

- 📚 [AI-Scientist-v2 Paper](https://pub.sakana.ai/ai-scientist-v2/paper)
- 🐙 [GitHub Repository](https://github.com/SakanaAI/AI-Scientist_v2)
- 📝 [Blog Post](https://sakana.ai/ai-scientist-first-publication/)
- ❓ [FAQ](https://github.com/SakanaAI/AI-Scientist_v2#frequently-asked-questions)

---
*Created for Kaggle - AI-Scientist-v2 Automated Scientific Discovery*